In [1]:
import pandas as pd
import numpy as np
from itertools import  product
import matplotlib.pyplot as plt

In [2]:
def get_n_qtr_fwd_bwd(cq,nq,forward=True,leading_zeros=True):
    cy = int(cq.split("FY")[1])
    cqn = int(cq[1])
    if leading_zeros:
        l = [
            f"Q{i % 4 + 1}FY{(cy + (i // 4)) % 100:02d}"  # Format year to have leading zeros
            for i in range(cqn, cqn + nq)
        ] if forward else [
            f"Q{i % 4 + 1}FY{(cy + (i // 4)) % 100:02d}"  # Format year to have leading zeros
            for i in range(cqn - nq, cqn - 1)
        ]
    else:
        l = [f"Q{i % 4 + 1}FY{cy + (i // 4)}" for i in range(cqn, cqn+nq)] if forward==True else [f"Q{i % 4 + 1}FY{cy + (i // 4)}" for i in range(cqn-nq,cqn-1)]  
    return l
def get_nth_qtr_fwd_bwd(cq, nq, forward=True,leading_zeros=True):
    cy = int(cq.split("FY")[1])
    cqn = int(cq[1])
    if not forward:
        nq = -nq
    new_qn = (cqn - 1 + nq) % 4 + 1
    new_cy = cy + (cqn - 1 + nq) // 4
    if leading_zeros:
        new_cy = f"{new_cy}" if new_cy >=10 else f"0{new_cy}"
    else:
        new_cy = f"{new_cy}"
    return f"Q{new_qn}FY{new_cy}"

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [3]:
sign_feat = ['TMR', 'Momentum','Beta','Alpha','Zscore','EP', 'BP', 'SP', 'Dividend_Flag', 'DE', 'EPS']

#### Price related features

In [4]:
df_tmr = pd.read_excel(r"D:\FY25-26\Quant Fund\data\qtrly_price_related_params_last_25_yrs_with_index\12M_returns_with_index_Last25Yrs.xlsx")

In [5]:
df_tmr.columns

Index(['Ticker', 'Industry', 'Q1FY00', 'Q2FY00', 'Q3FY00', 'Q4FY00', 'Q1FY01',
       'Q2FY01', 'Q3FY01', 'Q4FY01',
       ...
       'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24', 'Q4FY24', 'Q1FY25', 'Q2FY25',
       'Q3FY25', 'Q4FY25', 'Q1FY26'],
      dtype='object', length=107)

In [6]:
df_mom = pd.read_excel(r"D:\FY25-26\Quant Fund\data\qtrly_price_related_params_last_25_yrs_with_index\momentum_all_stocks_and_index_Last25Yrs.xlsx")

In [7]:
df_mom.columns

Index(['Ticker', 'Industry', 'Q1FY00', 'Q2FY00', 'Q3FY00', 'Q4FY00', 'Q1FY01',
       'Q2FY01', 'Q3FY01', 'Q4FY01',
       ...
       'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24', 'Q4FY24', 'Q1FY25', 'Q2FY25',
       'Q3FY25', 'Q4FY25', 'Q1FY26'],
      dtype='object', length=107)

In [8]:
df_beta = pd.read_excel(r"D:\FY25-26\Quant Fund\data\qtrly_price_related_params_last_25_yrs_with_index\beta_all_stocks_and_index_Last25Yrs.xlsx")
df_beta.columns

Index(['Unnamed: 0', 'Q1FY00', 'Q2FY00', 'Q3FY00', 'Q4FY00', 'Q1FY01',
       'Q2FY01', 'Q3FY01', 'Q4FY01', 'Q1FY02',
       ...
       'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24', 'Q4FY24', 'Q1FY25', 'Q2FY25',
       'Q3FY25', 'Q4FY25', 'Q1FY26'],
      dtype='object', length=106)

In [9]:
# df_beta['Unnamed: 0']
df_beta.rename(columns={'Unnamed: 0':'Ticker'},inplace=True)

In [10]:
df_alpha = pd.read_excel(r"D:\FY25-26\Quant Fund\data\qtrly_price_related_params_last_25_yrs_with_index\alpha_all_stocks_Last25Yrs.xlsx")
df_alpha.columns

Index(['Unnamed: 0', 'Q1FY00', 'Q2FY00', 'Q3FY00', 'Q4FY00', 'Q1FY01',
       'Q2FY01', 'Q3FY01', 'Q4FY01', 'Q1FY02',
       ...
       'Q3FY23', 'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24', 'Q4FY24', 'Q1FY25',
       'Q2FY25', 'Q3FY25', 'Q4FY25'],
      dtype='object', length=105)

In [11]:
# df_alpha['Unnamed: 0']
df_alpha.rename(columns={'Unnamed: 0':'Ticker'},inplace=True)

In [12]:
df_zscore = pd.read_excel(r"D:\FY25-26\Quant Fund\data\qtrly_price_related_params_last_25_yrs_with_index\z_score_stocks_and_index_Last25Yrs.xlsx")

In [13]:
df_zscore.columns

Index(['Ticker', 'Industry', 'Q1FY00', 'Q2FY00', 'Q3FY00', 'Q4FY00', 'Q1FY01',
       'Q2FY01', 'Q3FY01', 'Q4FY01',
       ...
       'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24', 'Q4FY24', 'Q1FY25', 'Q2FY25',
       'Q3FY25', 'Q4FY25', 'Q1FY26'],
      dtype='object', length=107)

In [14]:
sign_feat

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'Zscore',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS']

#### Stock related features

In [15]:
df_pe = pd.read_excel(r"D:\FY25-26\Quant Fund\data\stock_params_last_20_yrs\P_E.xlsx")
df_pe.columns

Index(['Ticker', 'Q4FY05', 'Q1FY06', 'Q2FY06', 'Q3FY06', 'Q4FY06', 'Q1FY07',
       'Q2FY07', 'Q3FY07', 'Q4FY07', 'Q1FY08', 'Q2FY08', 'Q3FY08', 'Q4FY08',
       'Q1FY09', 'Q2FY09', 'Q3FY09', 'Q4FY09', 'Q1FY10', 'Q2FY10', 'Q3FY10',
       'Q4FY10', 'Q1FY11', 'Q2FY11', 'Q3FY11', 'Q4FY11', 'Q1FY12', 'Q2FY12',
       'Q3FY12', 'Q4FY12', 'Q1FY13', 'Q2FY13', 'Q3FY13', 'Q4FY13', 'Q1FY14',
       'Q2FY14', 'Q3FY14', 'Q4FY14', 'Q1FY15', 'Q2FY15', 'Q3FY15', 'Q4FY15',
       'Q1FY16', 'Q2FY16', 'Q3FY16', 'Q4FY16', 'Q1FY17', 'Q2FY17', 'Q3FY17',
       'Q4FY17', 'Q1FY18', 'Q2FY18', 'Q3FY18', 'Q4FY18', 'Q1FY19', 'Q2FY19',
       'Q3FY19', 'Q4FY19', 'Q1FY20', 'Q2FY20', 'Q3FY20', 'Q4FY20', 'Q1FY21',
       'Q2FY21', 'Q3FY21', 'Q4FY21', 'Q1FY22', 'Q2FY22', 'Q3FY22', 'Q4FY22',
       'Q1FY23', 'Q2FY23', 'Q3FY23', 'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24',
       'Q4FY24', 'Q1FY25', 'Q2FY25', 'Q3FY25'],
      dtype='object')

In [16]:
df_div = pd.read_excel(r"D:\FY25-26\Quant Fund\data\stock_params_last_20_yrs\Dividend.xlsx")
df_div.columns

Index(['Ticker', 'Q4FY05', 'Q1FY06', 'Q2FY06', 'Q3FY06', 'Q4FY06', 'Q1FY07',
       'Q2FY07', 'Q3FY07', 'Q4FY07', 'Q1FY08', 'Q2FY08', 'Q3FY08', 'Q4FY08',
       'Q1FY09', 'Q2FY09', 'Q3FY09', 'Q4FY09', 'Q1FY10', 'Q2FY10', 'Q3FY10',
       'Q4FY10', 'Q1FY11', 'Q2FY11', 'Q3FY11', 'Q4FY11', 'Q1FY12', 'Q2FY12',
       'Q3FY12', 'Q4FY12', 'Q1FY13', 'Q2FY13', 'Q3FY13', 'Q4FY13', 'Q1FY14',
       'Q2FY14', 'Q3FY14', 'Q4FY14', 'Q1FY15', 'Q2FY15', 'Q3FY15', 'Q4FY15',
       'Q1FY16', 'Q2FY16', 'Q3FY16', 'Q4FY16', 'Q1FY17', 'Q2FY17', 'Q3FY17',
       'Q4FY17', 'Q1FY18', 'Q2FY18', 'Q3FY18', 'Q4FY18', 'Q1FY19', 'Q2FY19',
       'Q3FY19', 'Q4FY19', 'Q1FY20', 'Q2FY20', 'Q3FY20', 'Q4FY20', 'Q1FY21',
       'Q2FY21', 'Q3FY21', 'Q4FY21', 'Q1FY22', 'Q2FY22', 'Q3FY22', 'Q4FY22',
       'Q1FY23', 'Q2FY23', 'Q3FY23', 'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24',
       'Q4FY24', 'Q1FY25', 'Q2FY25', 'Q3FY25'],
      dtype='object')

In [17]:
df_pb = pd.read_excel(r"D:\FY25-26\Quant Fund\data\stock_params_last_20_yrs\P_B.xlsx")
df_pb.columns 

Index(['Ticker', 'Q4FY05', 'Q1FY06', 'Q2FY06', 'Q3FY06', 'Q4FY06', 'Q1FY07',
       'Q2FY07', 'Q3FY07', 'Q4FY07', 'Q1FY08', 'Q2FY08', 'Q3FY08', 'Q4FY08',
       'Q1FY09', 'Q2FY09', 'Q3FY09', 'Q4FY09', 'Q1FY10', 'Q2FY10', 'Q3FY10',
       'Q4FY10', 'Q1FY11', 'Q2FY11', 'Q3FY11', 'Q4FY11', 'Q1FY12', 'Q2FY12',
       'Q3FY12', 'Q4FY12', 'Q1FY13', 'Q2FY13', 'Q3FY13', 'Q4FY13', 'Q1FY14',
       'Q2FY14', 'Q3FY14', 'Q4FY14', 'Q1FY15', 'Q2FY15', 'Q3FY15', 'Q4FY15',
       'Q1FY16', 'Q2FY16', 'Q3FY16', 'Q4FY16', 'Q1FY17', 'Q2FY17', 'Q3FY17',
       'Q4FY17', 'Q1FY18', 'Q2FY18', 'Q3FY18', 'Q4FY18', 'Q1FY19', 'Q2FY19',
       'Q3FY19', 'Q4FY19', 'Q1FY20', 'Q2FY20', 'Q3FY20', 'Q4FY20', 'Q1FY21',
       'Q2FY21', 'Q3FY21', 'Q4FY21', 'Q1FY22', 'Q2FY22', 'Q3FY22', 'Q4FY22',
       'Q1FY23', 'Q2FY23', 'Q3FY23', 'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24',
       'Q4FY24', 'Q1FY25', 'Q2FY25', 'Q3FY25'],
      dtype='object')

In [18]:
df_sp = pd.read_excel(r"D:\FY25-26\Quant Fund\data\qtrly_peg_ev_ebitda_ratios_last_25_yrs_with_index\Price_to_Sales_Values_last_20yrs.xlsx")
df_sp.columns

Index(['Ticker', 'Q3FY05', 'Q4FY05', 'Q1FY06', 'Q2FY06', 'Q3FY06', 'Q4FY06',
       'Q1FY07', 'Q2FY07', 'Q3FY07', 'Q4FY07', 'Q1FY08', 'Q2FY08', 'Q3FY08',
       'Q4FY08', 'Q1FY09', 'Q2FY09', 'Q3FY09', 'Q4FY09', 'Q1FY10', 'Q2FY10',
       'Q3FY10', 'Q4FY10', 'Q1FY11', 'Q2FY11', 'Q3FY11', 'Q4FY11', 'Q1FY12',
       'Q2FY12', 'Q3FY12', 'Q4FY12', 'Q1FY13', 'Q2FY13', 'Q3FY13', 'Q4FY13',
       'Q1FY14', 'Q2FY14', 'Q3FY14', 'Q4FY14', 'Q1FY15', 'Q2FY15', 'Q3FY15',
       'Q4FY15', 'Q1FY16', 'Q2FY16', 'Q3FY16', 'Q4FY16', 'Q1FY17', 'Q2FY17',
       'Q3FY17', 'Q4FY17', 'Q1FY18', 'Q2FY18', 'Q3FY18', 'Q4FY18', 'Q1FY19',
       'Q2FY19', 'Q3FY19', 'Q4FY19', 'Q1FY20', 'Q2FY20', 'Q3FY20', 'Q4FY20',
       'Q1FY21', 'Q2FY21', 'Q3FY21', 'Q4FY21', 'Q1FY22', 'Q2FY22', 'Q3FY22',
       'Q4FY22', 'Q1FY23', 'Q2FY23', 'Q3FY23', 'Q4FY23', 'Q1FY24', 'Q2FY24',
       'Q3FY24', 'Q4FY24', 'Q1FY25', 'Q2FY25', 'Q3FY25', 'Q4FY25', 'Q1FY26'],
      dtype='object')

In [19]:
df_eps = pd.read_excel(r"D:\FY25-26\Quant Fund\data\qtrly_peg_ev_ebitda_ratios_last_25_yrs_with_index\EPS_Values_last_20yrs.xlsx")
df_eps.columns

Index(['Ticker', 'Q3FY05', 'Q4FY05', 'Q1FY06', 'Q2FY06', 'Q3FY06', 'Q4FY06',
       'Q1FY07', 'Q2FY07', 'Q3FY07', 'Q4FY07', 'Q1FY08', 'Q2FY08', 'Q3FY08',
       'Q4FY08', 'Q1FY09', 'Q2FY09', 'Q3FY09', 'Q4FY09', 'Q1FY10', 'Q2FY10',
       'Q3FY10', 'Q4FY10', 'Q1FY11', 'Q2FY11', 'Q3FY11', 'Q4FY11', 'Q1FY12',
       'Q2FY12', 'Q3FY12', 'Q4FY12', 'Q1FY13', 'Q2FY13', 'Q3FY13', 'Q4FY13',
       'Q1FY14', 'Q2FY14', 'Q3FY14', 'Q4FY14', 'Q1FY15', 'Q2FY15', 'Q3FY15',
       'Q4FY15', 'Q1FY16', 'Q2FY16', 'Q3FY16', 'Q4FY16', 'Q1FY17', 'Q2FY17',
       'Q3FY17', 'Q4FY17', 'Q1FY18', 'Q2FY18', 'Q3FY18', 'Q4FY18', 'Q1FY19',
       'Q2FY19', 'Q3FY19', 'Q4FY19', 'Q1FY20', 'Q2FY20', 'Q3FY20', 'Q4FY20',
       'Q1FY21', 'Q2FY21', 'Q3FY21', 'Q4FY21', 'Q1FY22', 'Q2FY22', 'Q3FY22',
       'Q4FY22', 'Q1FY23', 'Q2FY23', 'Q3FY23', 'Q4FY23', 'Q1FY24', 'Q2FY24',
       'Q3FY24', 'Q4FY24', 'Q1FY25', 'Q2FY25', 'Q3FY25', 'Q4FY25'],
      dtype='object')

In [20]:
df_de = pd.read_excel(r"D:\FY25-26\Quant Fund\data\stock_params_last_20_yrs\D_E.xlsx")
df_de.columns

Index(['Ticker', 'Q4FY05', 'Q1FY06', 'Q2FY06', 'Q3FY06', 'Q4FY06', 'Q1FY07',
       'Q2FY07', 'Q3FY07', 'Q4FY07', 'Q1FY08', 'Q2FY08', 'Q3FY08', 'Q4FY08',
       'Q1FY09', 'Q2FY09', 'Q3FY09', 'Q4FY09', 'Q1FY10', 'Q2FY10', 'Q3FY10',
       'Q4FY10', 'Q1FY11', 'Q2FY11', 'Q3FY11', 'Q4FY11', 'Q1FY12', 'Q2FY12',
       'Q3FY12', 'Q4FY12', 'Q1FY13', 'Q2FY13', 'Q3FY13', 'Q4FY13', 'Q1FY14',
       'Q2FY14', 'Q3FY14', 'Q4FY14', 'Q1FY15', 'Q2FY15', 'Q3FY15', 'Q4FY15',
       'Q1FY16', 'Q2FY16', 'Q3FY16', 'Q4FY16', 'Q1FY17', 'Q2FY17', 'Q3FY17',
       'Q4FY17', 'Q1FY18', 'Q2FY18', 'Q3FY18', 'Q4FY18', 'Q1FY19', 'Q2FY19',
       'Q3FY19', 'Q4FY19', 'Q1FY20', 'Q2FY20', 'Q3FY20', 'Q4FY20', 'Q1FY21',
       'Q2FY21', 'Q3FY21', 'Q4FY21', 'Q1FY22', 'Q2FY22', 'Q3FY22', 'Q4FY22',
       'Q1FY23', 'Q2FY23', 'Q3FY23', 'Q4FY23', 'Q1FY24', 'Q2FY24', 'Q3FY24',
       'Q4FY24', 'Q1FY25', 'Q2FY25', 'Q3FY25'],
      dtype='object')

In [21]:
df_div

,Ticker,Q4FY05,Q1FY06,Q2FY06,Q3FY06,Q4FY06,Q1FY07,Q2FY07,Q3FY07,Q4FY07,...,Q2FY23,Q3FY23,Q4FY23,Q1FY24,Q2FY24,Q3FY24,Q4FY24,Q1FY25,Q2FY25,Q3FY25
0,NSEMCAP,NaN,NaN,1.5677,NaN,1.1392,1.4115,1.1944,1.0595,1.3854,...,1.3470,1.2122,1.5010,1.2644,0.9607,0.8403,0.7649,0.689191,0.7033,0.008084
1,RELIANCE IN,2.0906,2.5380,2.0551,NaN,1.0351,1.0357,0.9368,0.8642,0.8811,...,0.3693,0.3447,0.3767,0.3443,0.3838,0.3482,0.3029,0.309459,0.3386,0.004068
2,TCS IN,0.4539,0.4786,0.8095,NaN,0.6265,0.7764,0.6604,0.6751,0.7902,...,1.4644,1.3818,3.5248,3.4825,3.2874,3.0843,1.7800,1.901809,1.7336,0.032723
3,HDFCB IN,0.6424,0.7075,0.6568,NaN,0.5812,0.6910,0.5944,0.5150,0.5764,...,1.0905,0.9520,0.9630,1.1167,1.2448,1.1116,1.3122,1.139451,1.1258,0.011272
4,BHARTI IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.3750,0.3722,0.4005,0.3414,0.4318,0.3875,0.3256,0.282696,0.4680,0.004844
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1035,YASHO IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0314,0.0315,0.0333,0.0274,0.0282,0.0307,0.0309,0.026312,0.0251,0.000273
1036,GOCL IN,2.7119,1.3316,1.3284,NaN,0.6315,0.7315,0.5675,0.5044,0.7649,...,1.0755,0.8192,1.0183,0.9608,2.1252,1.9286,2.1997,2.108148,0.9498,0.012270
1037,MBAP IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0551,0.0456,0.0440,0.0422,0.0818,0.0846,0.1036,NaN,NaN,NaN
1038,JIND IN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
df_div1 = (~df_div.set_index(keys=['Ticker']).isna()).astype(int).reset_index()

In [23]:
df_div1.head()

,Ticker,Q4FY05,Q1FY06,Q2FY06,Q3FY06,Q4FY06,Q1FY07,Q2FY07,Q3FY07,Q4FY07,...,Q2FY23,Q3FY23,Q4FY23,Q1FY24,Q2FY24,Q3FY24,Q4FY24,Q1FY25,Q2FY25,Q3FY25
0,NSEMCAP,0,0,1,0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
1,RELIANCE IN,1,1,1,0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
2,TCS IN,1,1,1,0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
3,HDFCB IN,1,1,1,0,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
4,BHARTI IN,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1


In [24]:
# data to be appended
# DE PE PB Dividend

In [25]:
df_pb1 = pd.read_excel(r"D:\FY25-26\Quant Fund\data\Q4FY25 Q1FY26 financial ratios business data.xlsx",usecols=['Ticker','P/B_Q4FY25'],skiprows=3).rename(columns={'P/B_Q4FY25':'Q4FY25'})
df_pe1 = pd.read_excel(r"D:\FY25-26\Quant Fund\data\Q4FY25 Q1FY26 financial ratios business data.xlsx",usecols=['Ticker','P/E_Q4FY25'],skiprows=3).rename(columns={'P/E_Q4FY25':'Q4FY25'})
df_div11 = pd.read_excel(r"D:\FY25-26\Quant Fund\data\Q4FY25 Q1FY26 financial ratios business data.xlsx",usecols=['Ticker','Dividend_Q4FY25'],skiprows=3).rename(columns={'Dividend_Q4FY25':'Q4FY25'})
df_de1 = pd.read_excel(r"D:\FY25-26\Quant Fund\data\Q4FY25 Q1FY26 financial ratios business data.xlsx",usecols=['Ticker','D/E_Q4FY25'],skiprows=3).rename(columns={'D/E_Q4FY25':'Q4FY25'})

In [26]:
df_pb1['Ticker'] = df_pb1['Ticker'].str.replace(" IN Equity","").str.replace(" Equity","").str.replace(" ","").str.lower()

In [27]:
df_pe1['Ticker'] = df_pe1['Ticker'].str.replace(" IN Equity","").str.replace(" Equity","").str.replace(" ","").str.lower()
df_div11['Ticker'] = df_div11['Ticker'].str.replace(" IN Equity","").str.replace(" Equity","").str.replace(" ","").str.lower()
df_de1['Ticker'] = df_de1['Ticker'].str.replace(" IN Equity","").str.replace(" Equity","").str.replace(" ","").str.lower()

In [28]:
df_pb['Ticker'] = df_pb['Ticker'].str.replace(" IN Equity","").str.replace(" IN","").str.lower()
df_pe['Ticker'] = df_pe['Ticker'].str.replace(" IN Equity","").str.replace(" IN","").str.lower()
df_div['Ticker'] = df_div['Ticker'].str.replace(" IN Equity","").str.replace(" IN","").str.lower()
df_de['Ticker'] = df_de['Ticker'].str.replace(" IN Equity","").str.replace(" IN","").str.lower()

In [29]:
df_pb[df_pb['Ticker']=='reliance']

,Ticker,Q4FY05,Q1FY06,Q2FY06,Q3FY06,Q4FY06,Q1FY07,Q2FY07,Q3FY07,Q4FY07,...,Q2FY23,Q3FY23,Q4FY23,Q1FY24,Q2FY24,Q3FY24,Q4FY24,Q1FY25,Q2FY25,Q3FY25
1,reliance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.8551,1.9873,2.0072,NaN,NaN,2.3282,NaN,2.669671,2.4383,2.438335


In [30]:
df_pb1[df_pb1['Ticker']=='reliance']

,Ticker,Q4FY25
1,reliance,2.174587


In [31]:
df_pb2 = df_pb.merge(df_pb1,how='left',on='Ticker')

In [32]:
df_pe2 = df_pe.merge(df_pe1,how='left',on='Ticker')

In [33]:
df_div2 = df_div.merge(df_div11,how='left',on='Ticker')

In [34]:
df_de2 = df_de.merge(df_de1,how='left',on='Ticker')

In [35]:
df_div2.head()

,Ticker,Q4FY05,Q1FY06,Q2FY06,Q3FY06,Q4FY06,Q1FY07,Q2FY07,Q3FY07,Q4FY07,...,Q3FY23,Q4FY23,Q1FY24,Q2FY24,Q3FY24,Q4FY24,Q1FY25,Q2FY25,Q3FY25,Q4FY25
0,nsemcap,NaN,NaN,1.5677,NaN,1.1392,1.4115,1.1944,1.0595,1.3854,...,1.2122,1.5010,1.2644,0.9607,0.8403,0.7649,0.689191,0.7033,0.008084,0.869113
1,reliance,2.0906,2.5380,2.0551,NaN,1.0351,1.0357,0.9368,0.8642,0.8811,...,0.3447,0.3767,0.3443,0.3838,0.3482,0.3029,0.309459,0.3386,0.004068,0.003921
2,tcs,0.4539,0.4786,0.8095,NaN,0.6265,0.7764,0.6604,0.6751,0.7902,...,1.3818,3.5248,3.4825,3.2874,3.0843,1.7800,1.901809,1.7336,0.032723,0.034386
3,hdfcb,0.6424,0.7075,0.6568,NaN,0.5812,0.6910,0.5944,0.5150,0.5764,...,0.9520,0.9630,1.1167,1.2448,1.1116,1.3122,1.139451,1.1258,0.011272,0.010666
4,bharti,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.3722,0.4005,0.3414,0.4318,0.3875,0.3256,0.282696,0.4680,0.004844,0.004615


In [36]:
df_de2.head()

,Ticker,Q4FY05,Q1FY06,Q2FY06,Q3FY06,Q4FY06,Q1FY07,Q2FY07,Q3FY07,Q4FY07,...,Q3FY23,Q4FY23,Q1FY24,Q2FY24,Q3FY24,Q4FY24,Q1FY25,Q2FY25,Q3FY25,Q4FY25
0,nsemcap,NaN,NaN,NaN,NaN,NaN,130.7728,130.9269,129.3907,117.6876,...,230.184,199.8450,195.0650,195.2505,193.8264,194.876,108.993537,114.6467,146.380119,140.536356
1,reliance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,40.3426,NaN,NaN,NaN,NaN,NaN,37.4637,37.463692,37.389422
2,tcs,NaN,NaN,NaN,NaN,0.9347,1.9814,NaN,NaN,NaN,...,NaN,8.2380,8.0462,NaN,NaN,NaN,9.325499,8.6555,8.655465,8.309341
3,hdfcb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,92.4359,86.5905,NaN,NaN,NaN,137.354792,133.9486,133.948579,126.117260
4,bharti,NaN,48.7099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,212.3367,218.2915,NaN,NaN,NaN,186.439145,204.4739,204.473858,135.125069


In [37]:
sign_feat

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'Zscore',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS']

In [38]:
df_alpha_fin = df_alpha[['Ticker','Q4FY25']].rename(columns={'Q4FY25':'Alpha'})
df_beta_fin = df_beta[['Ticker','Q4FY25']].rename(columns={'Q4FY25':'Beta'})
df_mom_fin = df_mom[['Ticker','Q4FY25']].rename(columns={'Q4FY25':'Momentum'})
df_tmr_fin = df_tmr[['Ticker','Q4FY25']].rename(columns={'Q4FY25':'TMR'})
df_zscore_fin = df_zscore[['Ticker','Q4FY25']].rename(columns={'Q4FY25':'Zscore'})

In [39]:
df_ep1 = 1/df_pe1.set_index(keys=['Ticker'])

In [40]:
df_ep_fin = df_ep1.reset_index().rename(columns={'Q4FY25':'EP'})

In [41]:
# df_pb1

df_bp1 = 1/df_pb1.set_index(keys=['Ticker'])
df_bp_fin = df_bp1.reset_index().rename(columns={'Q4FY25':'BP'})

In [42]:
df_sp1 = df_sp.set_index(keys=['Ticker'])[['Q4FY25']]

In [43]:
df_sp1 = 1/df_sp1

In [44]:
df_sp_fin = df_sp1.reset_index().rename(columns={'Q4FY25':'SP'})

In [45]:
df_div12 = (df_div11.set_index(keys=['Ticker'])>0).astype(int)
df_div_fin = df_div12.reset_index().rename(columns={'Q4FY25':'Dividend'})

In [46]:
df_de1.rename(columns={'Q4FY25':'DE'},inplace=True)

In [47]:
df_de_fin = df_de1.copy()

In [48]:
df_eps1 = df_eps[['Ticker','Q4FY25']]
df_eps_fin = df_eps1.rename(columns={'Q4FY25':'EPS'})

In [49]:
len(sign_feat)

11

In [50]:
sign_feat

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'Zscore',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS']

In [51]:
df_list = [df_bp_fin,df_de_fin,df_ep_fin,df_sp_fin,df_div_fin,df_eps_fin,df_mom_fin,df_tmr_fin,df_beta_fin,df_alpha_fin,df_zscore_fin]
for df in df_list:
    df['Ticker'] = df['Ticker'].str.replace(" ","").str.lower()
    df.set_index(keys=['Ticker'],inplace=True)

In [52]:
df_d_oot = pd.concat(df_list,axis=1)

In [53]:
df_d_oot.shape

(1075, 11)

In [54]:
df_d_oot.reset_index(inplace=True)

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os
from scipy.stats import skew

In [56]:
best_transforms = {'TMR': 'yeo-johnson',
 'Momentum': 'yeo-johnson',
 'Beta': 'yeo-johnson',
 'Alpha': 'yeo-johnson',
 'EP': 'boxcox',
 'BP': 'yeo-johnson',
 'SP': 'boxcox',
 'Dividend_Flag': 'original',
 'DE': 'yeo-johnson',
 'EPS': 'yeo-johnson',
 'Zscore': 'yeo-johnson'}

In [57]:
df_d_oot.replace([np.inf, -np.inf], 0, inplace=True)

In [58]:
df_d_oot['EP'] = df_d_oot['EP'].fillna(0)

In [59]:
df_d_oot['SP'].max()

np.float64(51.282051282051285)

In [60]:
df_d_oot['SP'] = df_d_oot['SP'].fillna(0)

In [61]:
sign_feat

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'Zscore',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS']

In [62]:
df_d_oot.columns

Index(['Ticker', 'BP', 'DE', 'EP', 'SP', 'Dividend', 'EPS', 'Momentum', 'TMR',
       'Beta', 'Alpha', 'Zscore'],
      dtype='object')

In [63]:
df_d_oot.rename(columns={'Dividend':'Dividend_Flag'},inplace=True)

In [64]:
import pickle 
filename = r'D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts\bi_lstm_top200_xform_label_encoder.pkl'
label_encoder = pickle.load(open(filename,'rb'))

In [65]:
label_encoder.get_params()

{'copy': True, 'with_mean': True, 'with_std': True}

In [66]:
df_d_oot

,Ticker,BP,DE,EP,SP,Dividend_Flag,EPS,Momentum,TMR,Beta,Alpha,Zscore
0,nsemcap,0.221794,140.536356,0.027053,0.000000,1.0,NaN,-5527.20,7.480903,1.000000,-1.387779e-17,NaN
1,reliance,0.459858,37.389422,0.040099,0.559065,1.0,14.3400,NaN,NaN,NaN,NaN,NaN
2,tcs,0.085543,8.309341,0.037417,0.195691,1.0,33.7858,-488.65,-6.969275,0.050093,-1.762377e-01,1.171788
3,hdfcb,0.359153,126.117260,0.049776,0.337701,1.0,24.6200,NaN,NaN,NaN,NaN,NaN
4,bharti,0.098084,135.125069,0.016262,0.173124,1.0,19.0200,145.65,41.087417,0.254908,6.793450e-02,1.776645
...,...,...,...,...,...,...,...,...,...,...,...,...
1070,arvindpo,NaN,NaN,0.000000,0.245809,NaN,NaN,-24.15,24.080268,1.261126,-1.637954e-01,-0.854146
1071,sip,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1072,sobhapp,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1073,grasimpp,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
label_encoder = LabelEncoder()
label_encoder.classes_ = np.load(r"D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts\bi_lstm_top200_xform_label_encoder.npy",allow_pickle=True)

In [68]:
label_encoder.classes_

array(['360one', '3m', 'aacl', 'aadharhf', 'aartipha', 'aavas', 'abb',
       'abcap', 'abfrl', 'abslamc', 'acc', 'acce', 'acem', 'aci', 'adani',
       'adaniens', 'adanigr', 'ade', 'adfl', 'adsez', 'advenzy',
       'aegislog', 'aether', 'affle', 'agig', 'agll', 'ahlu', 'aiae',
       'aiil', 'aisg', 'ajp', 'akzo', 'al', 'albc', 'alkem', 'alok',
       'alpm', 'amber', 'anandrat', 'anup', 'anuras', 'apat', 'apco',
       'aphs', 'apnt', 'apolp', 'apr', 'aptus', 'apty', 'arbp', 'arcp',
       'areal', 'arenm', 'arjn', 'arlf', 'artd', 'artmsl', 'arvindfa',
       'arvsmart', 'asbl', 'asfi', 'askautol', 'asmn', 'astel', 'asterdm',
       'astm', 'astra', 'atgl', 'atlp', 'atxl', 'aubank', 'aups',
       'avalon', 'avnt', 'awfis', 'axet', 'axsb', 'azad', 'baf',
       'bajajcon', 'bajel', 'baluforg', 'bandhan', 'barbeque', 'basf',
       'bata', 'bbox', 'bbtc', 'bcg', 'bchi', 'bcorp', 'bd', 'bde', 'bdl',
       'bectors', 'beml', 'bepl', 'bfin', 'bfut', 'bharti', 'bhartihe',
       'bhe',

In [69]:
#  filtering out unknown labels - that were not training by encoder

In [70]:
df_d_oot2 = df_d_oot[np.where(df_d_oot['Ticker'].isin(label_encoder.classes_),True,False)]

In [71]:
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np


def transform_feature(col, x, method):
    if method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        return pt.fit_transform(x.values.reshape(-1, 1)).flatten()
    elif method == 'boxcox':
        # Box-Cox requires strictly positive values
        # Add a small constant if min(x) == 0
        x_pos = x.copy()
        if (x_pos <= 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return boxcox(x_pos)[0]
    elif method == 'log1p':
        x_pos = x.copy()
        if (x_pos < 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return np.log1p(x_pos)
    else:
        return x.values

# 2. Apply transforms to train and test
df_d_oot_trans = df_d_oot2.copy()
# df_test_trans = df_test.copy()

for col in sign_feat:
    method = best_transforms.get(col, 'none')
    if method != 'none':
        df_d_oot_trans[col] = transform_feature(col, df_d_oot2[col], method)
        # df_test_trans[col] = transform_feature(col, df_test[col], method)

# 3. Scaling
# scaler = StandardScaler()
import pickle 
filename = r'D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts\bi_lstm_top200_xform_standard_scaler.pkl'
scaler = pickle.load(open(filename,'rb'))

train_data = scaler.fit_transform(df_d_oot_trans[sign_feat])
# test_data = scaler.transform(df_test_trans[sign_feat])

df_d_oot1 = pd.DataFrame(train_data, columns=sign_feat)
# df_test1 = pd.DataFrame(test_data, columns=sign_feat)

# for col in ['Ticker', 'Target', 'Quarter']:
#     df_d_oot1[col] = df_d_oot[col]
    # df_test1[col] = df_test[col]

# 4. Prepare X_train, y_train, X_test, y_test
features = sign_feat.copy()
X_train = df_d_oot1[features].values
# y_train = df_d_oot1['Target'].values
# X_test = df_test1[features].values
# y_test = df_test1['Target'].values

# If your model expects 3D input (samples, timesteps, features):
# X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
# X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train shape:", X_train.shape)
# print("y_train shape:", y_train.shape)
# print("X_test shape:", X_test.shape)
# print("y_test shape:", y_test.shape)
# ...existing code...

X_train shape: (942, 11)


In [72]:
df_d_oot2.reset_index(drop=False,inplace=True)

In [73]:
df_d_oot2['Ticker_Encoded'] = label_encoder.transform(df_d_oot2['Ticker'])

C:\Users\35098\AppData\Local\Temp\ipykernel_27844\137447066.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Ticker_Encoded'] = label_encoder.transform(df_d_oot2['Ticker'])


In [74]:
def create_ticker_embedding(input_dim, output_dim):
    return Embedding(input_dim=input_dim, output_dim=output_dim, input_length=1)

In [75]:
# Now, the LabelEncoder is aware of all the possible tickers in all datasets
embedding_size = len(label_encoder.classes_)  # Size of the embedding

# You can now proceed with creating your ticker embedding layer and other steps
ticker_embedding_layer = create_ticker_embedding(input_dim=embedding_size, output_dim=10)


d:\Quant Fund\qf_33\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [76]:
ticker_encoded_oot = df_d_oot2['Ticker_Encoded'].values

In [77]:

def calculate_metrics(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':f'{confusion}'
    }


def process_and_evaluate1(model, X_train, y_train,X_test, y_test, ticker_encoded_train,ticker_encoded_test,epochs,checkpoint_path,batch_size,k=5):
    
    
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    for train_index, val_index in kf.split(X_train):
        X_train1, X_val = X_train[train_index], X_train[val_index]
        y_train1, y_val = y_train[train_index], y_train[val_index]
        ticker_train, ticker_val = ticker_encoded_train[train_index], ticker_encoded_train[val_index]

    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                    save_weights_only=False,save_freq=epochs*batch_size,
                                                    verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=7, min_lr=1e-6, verbose=1)
    history = model.fit([ticker_train, X_train1], y_train1,
            validation_data=([ticker_val, X_val], y_val),
            epochs=epochs, batch_size=batch_size,
            callbacks=[early_stop, reduce_lr, cp_callback], verbose=0)

    # history = model.fit([ticker_encoded_train, X_train], y_train, epochs=epochs, batch_size=batch_size, validation_data=([ticker_encoded_test, X_test], y_test),callbacks=[cp_callback,early_stop])
    # model.load_weights(checkpoint_path)

    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='val')
    plt.legend()
    

### Bi-LSTM

In [78]:
from tensorflow.keras.layers import Bidirectional

def build_bi_lstm_top200_xform_model(input_shape, embedding_size, num_tickers):
    ticker_input = Input(shape=(1,), name='ticker_input')
    ticker_embedding = Embedding(input_dim=num_tickers, output_dim=embedding_size, input_length=1)(ticker_input)
    ticker_embedding_flat = Flatten()(ticker_embedding)
    
    continuous_input = Input(shape=(input_shape[1],), name='continuous_input')
    merged = Concatenate()([ticker_embedding_flat, continuous_input])
    reshaped_input = tf.keras.layers.Reshape((1, -1))(merged)
    bi_lstm_layer = Bidirectional(LSTM(16, activation='relu',dropout=0.2))(reshaped_input)
    output = Dense(1, activation='sigmoid')(bi_lstm_layer)    
    model = Model(inputs=[ticker_input, continuous_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [79]:
num_tickers = len(label_encoder.classes_)

In [80]:

# Define bi_lstm model
bi_lstm_top200_xform_model = build_bi_lstm_top200_xform_model(X_train.shape, embedding_size, num_tickers)
bi_lstm_top200_xform_model.compile(optimizer='adam', 
                   loss='binary_crossentropy',  # Binary classification problem
                   metrics=[AUC(),Precision()])

In [81]:
checkpoints_path = r"D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts"

In [82]:
bi_lstm_top200_xform_model_loaded = tf.keras.models.load_model(rf'{checkpoints_path}\bi_lstm_top200_xform.keras')

In [83]:
pred_probs_oot = bi_lstm_top200_xform_model_loaded.predict([ticker_encoded_oot, X_train])

30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [84]:
np.isnan(pred_probs_oot).sum()

np.int64(434)

In [88]:
preds_oot = pred_probs_oot.reshape(-1)
preds_oot = preds_oot>=0.5
preds_oot = pd.Series(preds_oot)
preds_oot = preds_oot.astype(int)

In [90]:
np.sum(preds_oot)

np.int64(43)

In [93]:
df_d_oot2['Pred_Tag'] = preds_oot
df_d_oot2['Pred_Prob'] = pred_probs_oot.reshape(-1)

C:\Users\35098\AppData\Local\Temp\ipykernel_27844\2128540863.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Tag'] = preds_oot
C:\Users\35098\AppData\Local\Temp\ipykernel_27844\2128540863.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Prob'] = pred_probs_oot.reshape(-1)


In [95]:
df_d_oot2['Pred_Prob'].fillna(0,inplace=True)

C:\Users\35098\AppData\Local\Temp\ipykernel_27844\2870514544.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_d_oot2['Pred_Prob'].fillna(0,inplace=True)
C:\Users\35098\AppData\Local\Temp\ipykernel_27844\2870514544.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Prob'].fillna(0,inplace=True)


In [96]:
df_d_oot2.head()

,index,Ticker,BP,DE,EP,SP,Dividend_Flag,EPS,Momentum,TMR,Beta,Alpha,Zscore,Ticker_Encoded,Pred_Tag,Pred_Prob
0,1,reliance,0.459858,37.389422,0.040099,0.559065,1.0,14.3400,NaN,NaN,NaN,NaN,NaN,698,0,0.000000
1,2,tcs,0.085543,8.309341,0.037417,0.195691,1.0,33.7858,-488.65,-6.969275,0.050093,-0.176238,1.171788,834,0,0.395408
2,3,hdfcb,0.359153,126.117260,0.049776,0.337701,1.0,24.6200,NaN,NaN,NaN,NaN,NaN,340,0,0.000000
3,4,bharti,0.098084,135.125069,0.016262,0.173124,1.0,19.0200,145.65,41.087417,0.254908,0.067935,1.776645,99,0,0.498063
4,5,icicibc,0.307441,96.004653,0.050907,0.309972,1.0,19.1100,NaN,NaN,NaN,NaN,NaN,371,0,0.000000


In [97]:
df_d_oot2.to_excel(r"D:\FY25-26\Quant Fund\Fund1\results\top200\3yrs_as_test_data_ts\Preds_as_of_end_of_Q4FY25.xlsx",index=False)

### Final Predictions - Significant Features

In [1]:
import pandas as pd
import numpy as np
from itertools import  product
import matplotlib.pyplot as plt

In [2]:
df_d_oot = pd.read_excel(r"D:\FY25-26\Quant Fund\Fund1\data\Q1FY26_as_input_data_22092025.xlsx")

In [3]:
df_d_oot.shape

(1072, 12)

In [4]:
df_d_oot.reset_index(inplace=True)

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os
from scipy.stats import skew

In [6]:
best_transforms = {'TMR': 'yeo-johnson',
 'Momentum': 'yeo-johnson',
 'Beta': 'yeo-johnson',
 'Alpha': 'yeo-johnson',
 'EP': 'boxcox',
 'BP': 'yeo-johnson',
 'SP': 'boxcox',
 'Dividend_Flag': 'original',
 'DE': 'yeo-johnson',
 'EPS': 'yeo-johnson',
 'Zscore': 'yeo-johnson'}

In [7]:
df_d_oot.replace([np.inf, -np.inf], 0, inplace=True)

In [8]:
df_d_oot['EP'] = df_d_oot['EP'].fillna(0)

In [9]:
df_d_oot['SP'].max()

np.float64(47.22108018040834)

In [10]:
df_d_oot['SP'] = df_d_oot['SP'].fillna(0)

In [12]:
# list of params being used in modelling
sign_feat = ['TMR', 'Momentum','Beta','Alpha','Zscore','EP', 'BP', 'SP', 'Dividend_Flag', 'DE', 'EPS']

In [13]:
df_d_oot.columns

Index(['index', 'Ticker', 'TMR', 'Momentum', 'Beta', 'Alpha', 'Zscore', 'EP',
       'BP', 'SP', 'Dividend_Flag', 'DE', 'EPS'],
      dtype='object')

In [15]:
import pickle 
filename = r'D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts\bi_lstm_top200_xform_label_encoder.pkl'
label_encoder = pickle.load(open(filename,'rb'))

In [16]:
label_encoder.get_params()

{'copy': True, 'with_mean': True, 'with_std': True}

In [17]:
df_d_oot

,index,Ticker,TMR,Momentum,Beta,Alpha,Zscore,EP,BP,SP,Dividend_Flag,DE,EPS
0,0,nsemcap,3.821191,7712.90,1.000000,0.000000,NaN,0.027746,0.230620,0.365956,1,123.896596,1694.871111
1,1,reliance,NaN,240.30,0.460970,0.110890,2.286165,0.033808,0.415244,0.475044,1,36.605139,51.470000
2,2,tcs,-15.964150,-165.05,0.493583,-0.127673,1.035974,0.038741,0.077346,0.203838,1,9.593757,134.190000
3,3,hdfcb,NaN,93.35,0.643362,0.000295,0.608351,0.046079,0.340684,0.306736,1,120.461093,46.405000
4,4,bharti,27.671233,293.70,0.309432,0.112026,1.460643,0.028629,0.092820,0.143449,1,139.209814,57.853350
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1067,1067,yasho,5.153643,347.35,0.085520,0.174375,0.100252,0.002676,0.166839,0.265806,1,138.803908,5.320000
1068,1068,gocl,-0.812113,85.90,0.200699,0.270101,-0.584623,0.000000,0.868493,0.305574,1,70.750344,31.710000
1069,1069,mbap,86.148884,146.20,-0.132187,0.521803,-0.557816,0.015673,0.110640,0.290175,1,78.833403,6.560000
1070,1070,jind,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.000000,0,69.159222,18.670000


In [18]:
# to remove column named index - which is intermittent - ignore if there is no such column
df_d_oot.drop(columns=['index'],inplace=True)

In [19]:
label_encoder = LabelEncoder()
label_encoder.classes_ = np.load(r"D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts\bi_lstm_top200_xform_label_encoder.npy",allow_pickle=True)

In [42]:
len(label_encoder.classes_)

942

In [20]:
#  filtering out unknown labels - that were not training by encoder

In [21]:
df_d_oot2 = df_d_oot[np.where(df_d_oot['Ticker'].isin(label_encoder.classes_),True,False)]

In [22]:
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np


def transform_feature(col, x, method):
    if method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        return pt.fit_transform(x.values.reshape(-1, 1)).flatten()
    elif method == 'boxcox':
        # Box-Cox requires strictly positive values
        # Add a small constant if min(x) == 0
        x_pos = x.copy()
        if (x_pos <= 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return boxcox(x_pos)[0]
    elif method == 'log1p':
        x_pos = x.copy()
        if (x_pos < 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return np.log1p(x_pos)
    else:
        return x.values

# 2. Apply transforms to train and test
df_d_oot_trans = df_d_oot2.copy()
# df_test_trans = df_test.copy()

for col in sign_feat:
    method = best_transforms.get(col, 'none')
    if method != 'none':
        df_d_oot_trans[col] = transform_feature(col, df_d_oot2[col], method)
        # df_test_trans[col] = transform_feature(col, df_test[col], method)

# 3. Scaling
# scaler = StandardScaler()
import pickle 
filename = r'D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts\bi_lstm_top200_xform_standard_scaler.pkl'
scaler = pickle.load(open(filename,'rb'))

train_data = scaler.fit_transform(df_d_oot_trans[sign_feat])
# test_data = scaler.transform(df_test_trans[sign_feat])

df_d_oot1 = pd.DataFrame(train_data, columns=sign_feat)
# df_test1 = pd.DataFrame(test_data, columns=sign_feat)

# for col in ['Ticker', 'Target', 'Quarter']:
#     df_d_oot1[col] = df_d_oot[col]
    # df_test1[col] = df_test[col]

# 4. Prepare X_train, y_train, X_test, y_test
features = sign_feat.copy()
X_train = df_d_oot1[features].values
# y_train = df_d_oot1['Target'].values
# X_test = df_test1[features].values
# y_test = df_test1['Target'].values

# If your model expects 3D input (samples, timesteps, features):
# X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
# X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train shape:", X_train.shape)
# print("y_train shape:", y_train.shape)
# print("X_test shape:", X_test.shape)
# print("y_test shape:", y_test.shape)
# ...existing code...

X_train shape: (963, 11)


In [23]:
df_d_oot2.reset_index(drop=False,inplace=True)

In [24]:
df_d_oot2['Ticker_Encoded'] = label_encoder.transform(df_d_oot2['Ticker'])

C:\Users\35098\AppData\Local\Temp\ipykernel_17236\137447066.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Ticker_Encoded'] = label_encoder.transform(df_d_oot2['Ticker'])


In [25]:
def create_ticker_embedding(input_dim, output_dim):
    return Embedding(input_dim=input_dim, output_dim=output_dim, input_length=1)

In [26]:
# Now, the LabelEncoder is aware of all the possible tickers in all datasets
embedding_size = len(label_encoder.classes_)  # Size of the embedding

# You can now proceed with creating your ticker embedding layer and other steps
ticker_embedding_layer = create_ticker_embedding(input_dim=embedding_size, output_dim=10)


d:\Quant Fund\qf_33\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [27]:
ticker_encoded_oot = df_d_oot2['Ticker_Encoded'].values

In [28]:

def calculate_metrics(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':f'{confusion}'
    }


def process_and_evaluate1(model, X_train, y_train,X_test, y_test, ticker_encoded_train,ticker_encoded_test,epochs,checkpoint_path,batch_size,k=5):
    
    
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    for train_index, val_index in kf.split(X_train):
        X_train1, X_val = X_train[train_index], X_train[val_index]
        y_train1, y_val = y_train[train_index], y_train[val_index]
        ticker_train, ticker_val = ticker_encoded_train[train_index], ticker_encoded_train[val_index]

    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                    save_weights_only=False,save_freq=epochs*batch_size,
                                                    verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=7, min_lr=1e-6, verbose=1)
    history = model.fit([ticker_train, X_train1], y_train1,
            validation_data=([ticker_val, X_val], y_val),
            epochs=epochs, batch_size=batch_size,
            callbacks=[early_stop, reduce_lr, cp_callback], verbose=0)

    # history = model.fit([ticker_encoded_train, X_train], y_train, epochs=epochs, batch_size=batch_size, validation_data=([ticker_encoded_test, X_test], y_test),callbacks=[cp_callback,early_stop])
    # model.load_weights(checkpoint_path)

    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='val')
    plt.legend()
    

### Bi-LSTM

In [29]:
from tensorflow.keras.layers import Bidirectional

def build_bi_lstm_top200_xform_model(input_shape, embedding_size, num_tickers):
    ticker_input = Input(shape=(1,), name='ticker_input')
    ticker_embedding = Embedding(input_dim=num_tickers, output_dim=embedding_size, input_length=1)(ticker_input)
    ticker_embedding_flat = Flatten()(ticker_embedding)
    
    continuous_input = Input(shape=(input_shape[1],), name='continuous_input')
    merged = Concatenate()([ticker_embedding_flat, continuous_input])
    reshaped_input = tf.keras.layers.Reshape((1, -1))(merged)
    bi_lstm_layer = Bidirectional(LSTM(16, activation='relu',dropout=0.2))(reshaped_input)
    output = Dense(1, activation='sigmoid')(bi_lstm_layer)    
    model = Model(inputs=[ticker_input, continuous_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [30]:
num_tickers = len(label_encoder.classes_)

In [31]:

# Define bi_lstm model
bi_lstm_top200_xform_model = build_bi_lstm_top200_xform_model(X_train.shape, embedding_size, num_tickers)
bi_lstm_top200_xform_model.compile(optimizer='adam', 
                   loss='binary_crossentropy',  # Binary classification problem
                   metrics=[AUC(),Precision()])

In [32]:
checkpoints_path = r"D:\FY25-26\Quant Fund\Fund1\src\3yrs_as_test_data_ts"

In [33]:
bi_lstm_top200_xform_model_loaded = tf.keras.models.load_model(rf'{checkpoints_path}\bi_lstm_top200_xform.keras')

In [34]:
pred_probs_oot = bi_lstm_top200_xform_model_loaded.predict([ticker_encoded_oot, X_train])

31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [35]:
np.isnan(pred_probs_oot).sum()

np.int64(221)

In [36]:
preds_oot = pred_probs_oot.reshape(-1)
preds_oot = preds_oot>=0.5
preds_oot = pd.Series(preds_oot)
preds_oot = preds_oot.astype(int)

In [37]:
np.sum(preds_oot)

np.int64(44)

In [38]:
df_d_oot2['Pred_Tag'] = preds_oot
df_d_oot2['Pred_Prob'] = pred_probs_oot.reshape(-1)

C:\Users\35098\AppData\Local\Temp\ipykernel_17236\2128540863.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Tag'] = preds_oot
C:\Users\35098\AppData\Local\Temp\ipykernel_17236\2128540863.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Prob'] = pred_probs_oot.reshape(-1)


In [39]:
df_d_oot2['Pred_Prob'].fillna(0,inplace=True)

C:\Users\35098\AppData\Local\Temp\ipykernel_17236\2870514544.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_d_oot2['Pred_Prob'].fillna(0,inplace=True)
C:\Users\35098\AppData\Local\Temp\ipykernel_17236\2870514544.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Prob'].fillna(0,inplace=True)


In [40]:
df_d_oot2.head()

,index,Ticker,TMR,Momentum,Beta,Alpha,Zscore,EP,BP,SP,Dividend_Flag,DE,EPS,Ticker_Encoded,Pred_Tag,Pred_Prob
0,1,reliance,NaN,240.30,0.460970,0.110890,2.286165,0.033808,0.415244,0.475044,1,36.605139,51.47000,698,0,0.000000
1,2,tcs,-15.964150,-165.05,0.493583,-0.127673,1.035974,0.038741,0.077346,0.203838,1,9.593757,134.19000,834,0,0.440834
2,3,hdfcb,NaN,93.35,0.643362,0.000295,0.608351,0.046079,0.340684,0.306736,1,120.461093,46.40500,340,0,0.000000
3,4,bharti,27.671233,293.70,0.309432,0.112026,1.460643,0.028629,0.092820,0.143449,1,139.209814,57.85335,99,0,0.386277
4,5,icicibc,NaN,113.85,0.700332,-0.024969,1.207945,0.049175,0.304835,0.285577,1,66.582003,72.41000,371,0,0.000000


In [ ]:
# df_d_oot2.to_excel(r"D:\FY25-26\Quant Fund\Fund1\results\top200\3yrs_as_test_data_ts\Preds_as_of_end_of_Q1FY26.xlsx",index=False)